# Multi-Seed Training for Arabic ITSM Classification

Addresses **reviewer Issue #2**: single-seed experiments insufficient.

Runs 4 key configurations × 3 seeds = 12 training runs on Kaggle T4 GPU.

**Estimated time**: ~2 GPU hours total (≈10 min per run × 12 runs)

**Configurations**:
1. MARBERTv2  L1-only (seeds: 42, 123, 456)
2. MARBERTv2  L1+L2+L3 (seeds: 42, 123, 456)
3. MARBERTv2  all-5-heads (seeds: 42, 123, 456)
4. AraBERTv2  L1+L2+L3 (seeds: 42, 123, 456)

Note: seed=42 runs replicate the original experiments to confirm reproducibility.

In [ ]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

In [ ]:
# Link the processed dataset
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
# Download models fully to local disk before training.
# This replaces HuggingFace lazy-streaming (which can stall at 30-40%) with
# a single complete download that all 12 training runs then load from local disk.
from huggingface_hub import snapshot_download

print('Downloading MARBERTv2 to local disk...')
MARBERT_PATH = snapshot_download(
    'UBC-NLP/MARBERTv2',
    local_dir='/kaggle/working/model_cache/marbert'
)
print('MARBERTv2 ready at:', MARBERT_PATH)

print('Downloading AraBERTv2 to local disk...')
ARABERT_PATH = snapshot_download(
    'aubmindlab/bert-base-arabertv02',
    local_dir='/kaggle/working/model_cache/arabert'
)
print('AraBERTv2 ready at:', ARABERT_PATH)


In [ ]:
import os
from pathlib import Path

METRICS_DIR = Path('results/metrics/multi_seed')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

# task_label matches what train.py computes:
#   --task l1         -> l1
#   --tasks l1 l2 l3  -> l1_l2_l3
#   --multi-task      -> multi_task
#
# model uses local paths (set in previous cell) to avoid HF network calls during training.
CONFIGS = [
    {'name': 'marbert_l1',     'model': MARBERT_PATH, 'flags': '--task l1',        'task_label': 'l1',        'epochs': 5},
    {'name': 'marbert_l1l2l3', 'model': MARBERT_PATH, 'flags': '--tasks l1 l2 l3', 'task_label': 'l1_l2_l3',  'epochs': 10},
    {'name': 'marbert_5heads', 'model': MARBERT_PATH, 'flags': '--multi-task',     'task_label': 'multi_task', 'epochs': 10},
    {'name': 'arabert_l1l2l3', 'model': ARABERT_PATH, 'flags': '--tasks l1 l2 l3', 'task_label': 'l1_l2_l3',  'epochs': 10},
]

print('Total runs:', len(CONFIGS) * len(SEEDS))
for cfg in CONFIGS:
    for seed in SEEDS:
        print(' ', cfg['name'], 'seed=' + str(seed))


In [ ]:
import json, os, time

ipy = get_ipython()
run_log = []

for cfg in CONFIGS:
    for seed in SEEDS:
        run_key = cfg['name'] + '_seed' + str(seed)
        out_metrics_path = METRICS_DIR / (run_key + '_metrics.json')

        if out_metrics_path.exists():
            print('[SKIP]', run_key, '-- already exists')
            continue

        seed_dir = 'models/seed_' + str(seed)
        checkpoint_dir = Path(seed_dir) / ('marbert_' + cfg['task_label'] + '_best')

        parts = ['python scripts/train.py', cfg['flags'],
                 '--model', cfg['model'],
                 '--epochs', str(cfg['epochs']),
                 '--seed', str(seed),
                 '--output-dir', seed_dir,
                 '--data-dir data/processed']
        cmd = ' '.join(parts)

        print('')
        print('[RUN]', run_key)
        print('  CMD:', cmd)
        print('  Checkpoint will be:', checkpoint_dir)
        t0 = time.time()
        ipy.system(cmd)
        elapsed = time.time() - t0
        print('  Time: %.1f min' % (elapsed/60,))

        test_metrics_file = checkpoint_dir / 'test_metrics.json'
        if test_metrics_file.exists():
            with open(test_metrics_file) as fp:
                test_metrics = json.load(fp)
            result = {'config': cfg['name'], 'seed': seed, 'metrics': test_metrics}
            with open(out_metrics_path, 'w') as fp:
                json.dump(result, fp, indent=2)
            print('  Saved:', out_metrics_path)
            run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed/60})
        else:
            print('  WARNING: test_metrics.json not found at', test_metrics_file)
            run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed/60})

print('')
print('=== Run log ===')
for r in run_log:
    print(' ', r['run'] + ':', r['status'], '(%.1f min)' % r['time_min'])


In [ ]:
# Aggregate results
!python scripts/aggregate_multi_seed.py --metrics-dir results/metrics/multi_seed

In [ ]:
# View summary
import json
with open('results/metrics/multi_seed_summary.json') as f:
    summary = json.load(f)

for config, data in summary['summary'].items():
    print(f'\n{config} (seeds: {data["seeds"]}):')
    for task, stats in data['tasks'].items():
        print(f'  {task}: {stats["mean"]*100:.2f} ± {stats["std"]*100:.2f} (F1%)')

In [ ]:
# Package all results for download
!zip -r multi_seed_results.zip results/metrics/multi_seed/ results/metrics/multi_seed_summary.json
print('Created multi_seed_results.zip — download from Kaggle output')